In [117]:
import os
from pandas import read_csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pandas import Timestamp
from daesim.utils import ODEModelSolver

from daesim.climate import *
from daesim.plantgrowthphases import PlantGrowthPhases
from daesim.management import ManagementModule
from daesim.plant_1000_thermaltime import PlantModuleCalculator

from daesim2_analysis.parameters import Parameters
from daesim2_analysis.forcing_data import ForcingData
os.chdir('/g/data/xe2/ya6227/NVTAnalysis')
from NVTAnalysis.canola_stage_sampler import CanolaStageSampler
from os.path import exists
from pandas import Timestamp
from daesim2_analysis.utils import load_df_forcing
from pandas import DataFrame

In [14]:
df_Hyola_Blazer_TT = read_csv('data/selected_10_Hyola_Blazer_TT.csv')

In [138]:
from PaddockTS.query import Query
from daesim2_analysis.daesim_config import DAESIMConfig
from daesim2_analysis.parameters import Parameters
from PaddockTS.Data.environmental import download_environmental_data
from daesim2_analysis.run import update_attribute, update_attribute_in_phase

In [74]:
queries: list[Query] = []

In [88]:
for idx, row in df_Hyola_Blazer_TT.iterrows():
    query = Query(
        lat=row['Trial GPS Lat'],
        lon=row['Trial GPS Long'],
        collections=['ga_s2am_ard_3', 'ga_s2bm_ard_3'],
        buffer=0.01,
        bands=[
            'nbart_blue',
            'nbart_green',
            'nbart_red',
            'nbart_red_edge_1',
            'nbart_red_edge_2',
            'nbart_red_edge_3',
            'nbart_nir_1',
            'nbart_nir_2',
            'nbart_swir_2',
            'nbart_swir_3'
        ],
        start_time=date.fromisoformat(row['SowingDate']),
        end_time=date.fromisoformat(row['HarvestDate']),
        out_dir='/g/data/xe2/ya6227/NVTAnalysis/data/DAESim',
        tmp_dir='/g/data/xe2/ya6227/NVTAnalysis/data/DAESim',
        stub=row['TrialCode']
    )
    queries += [query]
    # download_environmental_data(query)

In [143]:
# parameters = Parameters.__from_file__("/g/data/xe2/ya6227/daesim2-analysis/parameters/Fast1.json")

parameters = Parameters(
    paths           = ["PlantDev", "PlantDev"],
    modules         = ["PlantDev", "PlantDev"],
    names           = ["gdd_requirements", "gdd_requirements"],
    units           = ["deg C d", "deg C d"],
    init            = [900, 650],
    min             = [600, 350],
    max             = [1800, 700],
    phase_specific  = [True, True],
    phase           = ["vegetative", "grainfill"]
)

In [77]:
daesim_config = DAESIMConfig.from_json_dict("/g/data/xe2/ya6227/daesim2-analysis/daesim_configs/DAESIM1.json") 

In [91]:
sampler = CanolaStageSampler(
    mean_fractions=(0.09, 0.36, 0.15, 0.40),  # tweak if you have cultivar priors
    kappa=30.0,                               # Higher the value, higher the number of times you sample values close to mean
    min_days=(30, 30, 30, 30),                # minimum days in each stage
    seed=123                                  
)

In [147]:
def get_input_data_from_query(query: Query):
    SiteX = ClimateModule(CLatDeg=query.lat,CLonDeg=query.lon,timezone=10)
    ForcingDataX = ForcingData(
        SiteX=SiteX,
        sowing_dates=[Timestamp(query.start_time)],
        harvest_dates=[Timestamp(query.end_time)],
        df=load_df_forcing(f'{query.stub_tmp_dir}/environmental/{query.stub}_DAESim_forcing.csv'),
        df_type='0',
        zero_crossing_indices=[0,1]
    )
    ManagementX = ManagementModule(cropType="Canola", sowingDays=ForcingDataX.sowing_days, harvestDays=ForcingDataX.harvest_days, sowingYears=ForcingDataX.sowing_years, harvestYears=ForcingDataX.harvest_years)
    PlantDevX = PlantGrowthPhases(
        phases=["germination", "vegetative", "anthesis", "grainfill", "maturity"],
        gdd_requirements=[120, 500, 200, 350, 200],
        vd_requirements=[0, 25, 0, 0, 0],
        allocation_coeffs=[
            [0.2, 0.1, 0.7, 0.0, 0.0],   # Phase 1
            [0.5, 0.1, 0.4, 0.0, 0.0],   # Phase 2
            [0.25, 0.5, 0.25, 0.0, 0.0], # Phase 3
            [0.1, 0.1, 0.1, 0.7, 0.0],   # Phase 4
            [0.1, 0.1, 0.1, 0.7, 0.0]    # Phase 5
        ],
        turnover_rates = [
            [0.001, 0.001, 0.001, 0.0, 0.0],  # Phase 1
            [0.01,  0.002, 0.01,  0.0, 0.0],  # Phase 2
            [0.02,  0.002, 0.04,  0.0, 0.0],  # Phase 3
            [0.10,  0.008, 0.10,  0.0, 0.0],  # Phase 4
            [0.50,  0.017, 0.50,  0.0, 0.0]   # Phase 5
        ]    ## Turnover rates per pool and developmental phase (days-1))
    )
    
    PlantX = PlantModuleCalculator(
        Site=SiteX,
        Management=ManagementX,
        PlantDev=PlantDevX,
        GDD_method="linear1",
        GDD_Tbase=0.0,
        GDD_Tupp=25.0,
    )
    
    # %%
    ## Define the callable calculator that defines the right-hand-side ODE function
    PlantXCalc = PlantX.calculate
    
    input_data = [
        ODEModelSolver,
        PlantX,
        ForcingDataX.time_axis,
        ForcingDataX.inputs,
        ForcingDataX.reset_days,
        ForcingDataX.zero_crossing_indices,
        ForcingDataX.time_nday_f,
        ForcingDataX.time_doy_f,
        ForcingDataX.time_year_f
    ]
    return input_data

In [192]:
def get_target_and_uncertainity_from_query(query: Query):
    observables_names = ["start of vegetative", "start of flowering", "start of grainfill", "harvest"]
    observables_units = ["ordinal day of year", "ordinal day of year", "ordinal day of year", "ordinal day of year"]
    synthetic_observables_df = sampler.sample(str(query.start_time), str(query.end_time), n=1)
    observables_values = synthetic_observables_df[['vegetative_start_doy', 'flowering_start_doy', 'grainfill_start_doy', 'harvest_doy']].iloc[0].tolist()
    # observables_values = CanolaStageSampler(str(query.start_time), str(query.end_time))
    observables_uncertainty = [5, 5, 5, 5]
    x = sampler.sample(str(query.start_time), str(query.end_time), n=1)
    target_df = pd.DataFrame({
        "Name": observables_names,
        "Units": observables_units,
        "Values": observables_values,
        "Uncertainty": observables_uncertainty,
    })
    y = target_df["Values"].values
    U_y = target_df["Uncertainty"].values
    return y, U_y

In [149]:
get_observables_for_query(query)

In [150]:
DAESims = [get_DAESim_from_query(query) for query in queries]

In [177]:
def run_model_and_get_outputs(Plant, ODEModelSolver, time_axis, forcing_inputs, reset_days, zero_crossing_indices):
    ## Define the callable calculator that defines the right-hand-side ODE function
    PlantCalc = Plant.calculate
    
    Model = ODEModelSolver(calculator=PlantCalc, states_init=[0.0, 0.0], time_start=time_axis[0], log_diagnostics=True)
    
    ## Run the model solver
    res = Model.run(
        time_axis=time_axis,
        forcing_inputs=forcing_inputs,
        solver="euler",
        zero_crossing_indices=zero_crossing_indices,
        reset_days=reset_days,
    )

    # Convert the defaultdict to a regular dictionary
    _diagnostics = dict(Model.diagnostics)
    # Convert each list in the dictionary to a NumPy array
    diagnostics = {key: np.array(value) for key, value in _diagnostics.items()}

    # Convert the array to a numeric type, handling mixed int and float types
    diagnostics['idevphase_numeric'] = np.array(diagnostics['idevphase'],dtype=np.float64)
    
    # In the model idevphase can equal None but that is not useable in post-processing, so we set None values to np.nan
    diagnostics["idevphase_numeric"][diagnostics["idevphase"] == None] = np.nan

    # Add np.nan to the end of each array in the dictionary to represent the last time point in the time_axis (corresponds to the last time point of the state vector)
    for key in diagnostics:
        if key == "t":
            diagnostics[key] = np.append(diagnostics[key], res["t"][-1])
        else:
            diagnostics[key] = np.append(diagnostics[key], np.nan)

    # Add state variables to the diagnostics dictionary
    diagnostics["GDD"] = res["y"][0,:]
    diagnostics["VD"] = res["y"][1,:]

    # Add forcing inputs to diagnostics dictionary
    for i,f in enumerate(forcing_inputs):
        ni = i+1
        if f(time_axis[0]).size == 1:
            fstr = f"forcing {ni:02}"
            diagnostics[fstr] = f(time_axis)
        elif f(time_axis[0]).size > 1:
            # this forcing input has levels/layers (e.g. multilayer soil moisture)
            nz = f(time_axis[0]).size
            for iz in range(nz):
                fstr = f"forcing {ni:02} z{iz}"
                diagnostics[fstr] = f(time_axis)[:,iz]
    
    # Observation Operator
    # Calculate model-equivalent observations from model run output

    # # Diagnose time indexes when developmental phase transitions occur
    ngrowing_seasons = (len(Plant.Management.sowingDays) if (isinstance(Plant.Management.sowingDays, int) == False) else 1)
    if ngrowing_seasons > 1:
        # print("Multiple sowing and harvest events occur. Only returning results for first growing season.")
        ## ignore any time steps before first sowing event and after last harvest event
        it_sowing = np.where(time_axis == reset_days[0])[0][0]  #sowing_steps_itax[0]
        
        if Plant.Management.harvestDays is not None:
            it_harvest = np.where(time_axis == reset_days[1])[0][0]  #harvest_steps_itax[0]   # np.where(np.floor(Climate_doy_f(time_axis)) == Plant.Management.harvestDay)[0][0]
        else:
            it_harvest = -1   # if there is no harvest day specified, we just take the last day of the simulation. 
    else:
        # print("Just one sowing event and one harvest event occurs. Returning results for first (and only) growing season.")
        ## ignore any time steps before first sowing event and after last harvest event
        it_sowing = np.where(time_axis == reset_days[0])[0][0]  #sowing_steps_itax[0]
        
        if Plant.Management.harvestDays is not None:
            it_harvest = np.where(time_axis == reset_days[1])[0][0]  #harvest_steps_itax[0]   # np.where(np.floor(Climate_doy_f(time_axis)) == Plant.Management.harvestDay)[0][0]
        else:
            it_harvest = -1   # if there is no harvest day specified, we just take the last day of the simulation. 

    # Diagnose time indexes when developmental phase transitions occur

    # Convert the array to a numeric type, handling mixed int and float types
    idevphase = diagnostics["idevphase_numeric"]   #[it_sowing:it_harvest+1]
    valid_mask = ~np.isnan(idevphase)
    
    # Identify all transitions (number-to-NaN, NaN-to-number, or number-to-different-number)
    it_phase_transitions = np.where(
        ~valid_mask[:-1] & valid_mask[1:] |  # NaN-to-number
        valid_mask[:-1] & ~valid_mask[1:] |  # Number-to-NaN
        (valid_mask[:-1] & valid_mask[1:] & (np.diff(idevphase) != 0))  # Number-to-different-number
    )[0] + 1
    
    # Time index for the end of the maturity phase
    if Plant.PlantDev.phases.index('maturity') in idevphase:
        it_mature = np.where(idevphase == Plant.PlantDev.phases.index('maturity'))[0][-1]    # Index for end of maturity phase
    elif Plant.Management.harvestDays is not None: 
        it_mature = it_harvest    # Maturity developmental phase not completed, so take harvest as the end of growing season
    else:
        it_mature = -1    # if there is no harvest day specified, we just take the last day of the simulation. 

    # it_sowing = np.where(time_axis == Plant.Management.sowingDay)[0][0]
    # if Plant.Management.harvestDay is not None:
    #     it_harvest = np.where(time_axis == Plant.Management.harvestDay)[0][0]
    # else:
    #     it_harvest = -1   # if there is no harvest day specified, we just take the last day of the simulation. 

    # # Convert the array to a numeric type, handling mixed int and float types
    # idevphase = diagnostics["idevphase_numeric"]
    # valid_mask = ~np.isnan(idevphase)
    
    # # Identify all transitions (number-to-NaN, NaN-to-number, or number-to-different-number)
    # it_phase_transitions = np.where(
    #     ~valid_mask[:-1] & valid_mask[1:] |  # NaN-to-number
    #     valid_mask[:-1] & ~valid_mask[1:] |  # Number-to-NaN
    #     (valid_mask[:-1] & valid_mask[1:] & (np.diff(idevphase) != 0))  # Number-to-different-number
    # )[0] + 1
    
    # # Time index for the end of the maturity phase
    # if Plant.PlantDev.phases.index('maturity') in idevphase:
    #     it_mature = np.where(idevphase == Plant.PlantDev.phases.index('maturity'))[0][-1]    # Index for end of maturity phase
    # elif Plant.Management.harvestDay is not None: 
    #     it_mature = it_harvest    # Maturity developmental phase not completed, so take harvest as the end of growing season
    # else:
    #     it_mature = -1    # if there is no harvest day specified, we just take the last day of the simulation. 

    # import pdb; pdb.set_trace()
    # Filter out transitions that occur on or before the sowing day
    # it_phase_transitions = [t for t in it_phase_transitions if time_axis[t] > time_axis[it_sowing+1]]
    it_phase_transitions = [t for t in it_phase_transitions if t > int(it_sowing+1)]
    # Filter out transitions that occur after the maturity or harvest day
    # it_phase_transitions = [t for t in it_phase_transitions if time_axis[t] <= time_axis[it_mature]]
    it_phase_transitions = [t for t in it_phase_transitions if t <= it_mature]

    # Developmental phase indexes
    igermination = Plant.PlantDev.phases.index("germination")
    ivegetative = Plant.PlantDev.phases.index("vegetative")
    if Plant.Management.cropType == "Wheat":
        ispike = Plant.PlantDev.phases.index("spike")
    ianthesis = Plant.PlantDev.phases.index("anthesis")
    igrainfill = Plant.PlantDev.phases.index("grainfill")
    imaturity = Plant.PlantDev.phases.index("maturity")

    ip = np.where(diagnostics['idevphase'][it_phase_transitions] == Plant.PlantDev.phases.index('vegetative'))[0][0]
    tdoy_vegetative = time_axis[it_phase_transitions[ip]]   # ordinal day-of-year at transition point into vegetative phase
    if Plant.PlantDev.phases.index('anthesis') in idevphase[it_sowing+1:it_harvest+1]:
        ip = np.where(diagnostics['idevphase'][it_phase_transitions] == Plant.PlantDev.phases.index('anthesis'))[0][0]
        tdoy_anth0 = time_axis[it_phase_transitions[ip]]   # ordinal day-of-year at transition point into anthesis phase
    else:
        tdoy_anth0 = time_axis[it_harvest]
    if Plant.PlantDev.phases.index('grainfill') in idevphase[it_sowing+1:it_harvest+1]:
        ip = np.where(diagnostics['idevphase'][it_phase_transitions] == Plant.PlantDev.phases.index('grainfill'))[0][0]
        tdoy_anth1 = time_axis[it_phase_transitions[ip]]   # ordinal day-of-year at transition point into grainfill stage (out of anthesis phase)
    else:
        tdoy_anth1 = time_axis[it_harvest]
    tdoy_harvest = time_axis[it_harvest]   # ordinal day-of-year at harvest
    
    # import pdb; pdb.set_trace()
    # Model output (of observables) given the parameter vector p
    # - this is the model output that we compare to observations and use to calibrate the parameters
    M_p = np.array([
        tdoy_vegetative, 
        tdoy_anth0, 
        tdoy_anth1,
        tdoy_harvest,
    ])

    return M_p

In [178]:
def objective_function_wls(params, observations, observation_unc_sigma, Plant, input_data, param_info):
    """
    Objective function using weighted least squares (WLS). 

    Notes
    -----
    The cost function, $J$, is defined using a weighted least-squares as follows: 
    
    $J = \sum_{i=1}^n \frac{(M_i(p) - y_i)^2}{\sigma^2}$
    
    Where $y$ is the vector of observations, $M$ is the vector model predicted observables 
    given parameter set $p$, and $\sigma$ is the observation errors (assumed to include 
    structural model errors). Note that this formulation ignores the priors. 
    """
    # Round the parameters off to integers as these parameters must be integers representing day-of-year
    int_params = np.round(params).astype(int)
    # Calculate model outputs
    model_outputs = model_function(int_params, Plant, input_data, param_info)
    # Calculate the error as the weighted Least Squares: 
    # 
    # Error is the model - observed difference squared, normalised by the uncertainty (as a variance), and summed over all obs
    error = np.mean( ((model_outputs - observations) ** 2) / (observation_unc_sigma**2))
    print(f"Current error: {error}")
    return error

In [179]:
def model_function(params: np.ndarray, params_info: DataFrame, query: Query):
    input_data = get_input_data_from_query(query)
    ODEModelSolver, model_instance, time_axis, forcing_inputs, reset_days, zero_crossing_indices, time_nday_f, time_doy_f, time_year_f = input_data
    for idx, value in enumerate(params):
        param_name = param_info["Name"].values[idx]
        param_path = param_info["Module Path"].values[idx]
        full_path = f"{param_path}.{param_name}"
        phase_specific = param_info["Phase Specific"].values[idx]
        
        if phase_specific:
            # Handle phase-specific parameters
            phase = param_info["Phase"].values[idx]
            update_attribute_in_phase(model_instance, full_path, value, phase)
        else:
            if (param_name == "sowingDays") or (param_name == "harvestDays"):
                # Update parameters that must be defined as a list type
                update_attribute(model_instance, full_path, [value])
            else:
                # Update regular parameters
                update_attribute(model_instance, full_path, value)

        # Make sure the solver knows about the sowing and harvest dates as well (to reset the state variables like GDD and VD)
        if (param_name == "sowingDays") or (param_name == "harvestDays"):
            # Find value of time_nday_f where time_doy_f == sowingDay and time_year_f == sowingYear.
            sowingDay, sowingYear = model_instance.Management.sowingDays, model_instance.Management.sowingYears
            sowing_nday = time_nday_f[(np.floor(time_doy_f) == sowingDay) & (np.array(time_year_f) == sowingYear)]
            
            # Find value of time_nday_f where time_doy_f == sowingDay and time_year_f == sowingYear.
            harvestDay, harvestYear = model_instance.Management.harvestDays, model_instance.Management.harvestYears
            harvest_nday = time_nday_f[(np.floor(time_doy_f) == harvestDay) & (np.array(time_year_f) == harvestYear)]
            
            # Set reset_days to be the updated sowing and harvest nday
            reset_days = [sowing_nday[0], harvest_nday[0]]
     
    model_output = run_model_and_get_outputs(model_instance, ODEModelSolver, time_axis, forcing_inputs, reset_days, zero_crossing_indices)

    return model_output

In [193]:
def objective_function(params: np.ndarray, params_info: DataFrame, queries: list[Query]):
    int_params = np.round(params).astype(int)
    errors = []
    for q in queries:
        
        model_outputs = model_function(int_params, param_info, query)
        observations, observations_unc_sigma = get_target_and_uncertainity_from_query(query)
        error = np.mean( ((model_outputs - observations) ** 2) / (observation_unc_sigma**2))
        errors += [error]

    return sum(errors)

In [194]:
param_info = parameters.df
params = parameters.df['Initial Value'].values

In [195]:
a = objective_function(params, param_info, queries)

NameError: name 'Y' is not defined

In [188]:
print(a)

None
